# scRareBench | High-Level MrVI | Dataset 5 — NYGC / Seurat v4 PBMC

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

**Pipeline:** `load_dataset(5) → MrVI MethodSpec → benchmark_method → scIB + rare-cell reports → ZIP`.

- **Dataset:** `nygc_seurat_v4_pbmc` (152,145 cells after scRareBench-owned official QC).
- **Raw RNA counts:** `adata.X` (no `counts` layer); **MrVI sample:** `orig.ident` (24 samples).
- **Method HVGs:** 4,000, Seurat-v3, sample-aware; **MrVI output:** 30-dimensional posterior-mean **u** latent (`give_z=False`).
- **Method seeds:** 42, 123, 2026; **fixed benchmark seed:** 42.
- **Evaluation labels:** `celltype.l2` are only used by scRareBench, not by MrVI. Dataset 5 rare-scenario annotations are provisional.

Select a **GPU** (preferably **High-RAM**) Colab runtime. Method-specific preprocessing/training remains in the adapter; scRareBench owns dataset QC/metadata, scIB, rare-cell evaluation and report generation.

## 1. Install pinned scRareBench and MrVI

In [ ]:
import subprocess
import sys

SCRAREBENCH_REVISION = "cfdb720f5488b9ceef9a225b35fc7558bcb0251c"  # v0.10.6
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    f"git+https://github.com/amirhossein-alishahi/scRareBench_.git@{SCRAREBENCH_REVISION}",
])

METHOD_DEPENDENCIES = ("scvi-tools==1.4.3",)
from scrarebench.runtime import setup_runtime, print_install_report

install_report = setup_runtime(
    extra_requirements=METHOD_DEPENDENCIES,
    extra_imports=("scvi", "scvi.external"),
    quiet=False,
)
print_install_report(install_report)

## 2. Load Dataset 5

In [ ]:
from scrarebench import load_dataset

adata = load_dataset(5)
print(adata)

## 3. Method and benchmark settings

In [ ]:
METHOD_SEEDS = [42, 123, 2026]
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    "sample_key": "orig.ident",
    "n_hvg": 4000,
    "hvg_span": 0.3,
    "n_latent_u": 30,
    "n_latent_z": 30,
    "encoder_n_hidden": 128,
    "encoder_n_layers": 2,
    "max_epochs": 300,
    "train_size": 0.90,
    "batch_size": 128,
    "early_stopping": True,
    "early_stopping_patience": 25,
}

OUTPUT_DIR = "/content/scrarebench_results/dataset5_mrvi"

## 4. MrVI method adapter

Only MrVI-specific work: select HVGs from unnormalized `adata.X`, build an **RNA-only** training AnnData containing **only sample metadata**, fit MrVI, and return the **u** embedding in original cell order. No biological labels are passed into MrVI.

In [ ]:
import gc

import anndata as ad
import numpy as np
import scanpy as sc
import scvi
import torch
from scvi.external import MRVI
from scrarebench import MethodOutput


def run_mrvi(method_adata, seed, config):
    sample_key = config["sample_key"]

    sc.pp.highly_variable_genes(
        method_adata,
        layer=None,
        flavor="seurat_v3",
        n_top_genes=config["n_hvg"],
        batch_key=sample_key,
        span=config["hvg_span"],
        subset=False,
    )
    hvg_mask = method_adata.var["highly_variable"].to_numpy(dtype=bool)

    mrvi_adata = ad.AnnData(
        X=method_adata.X[:, hvg_mask].astype(np.float32),
        obs=method_adata.obs[[sample_key]].copy(),
        var=method_adata.var.loc[hvg_mask].copy(),
    )

    scvi.settings.seed = int(seed)
    MRVI.setup_anndata(
        mrvi_adata,
        layer=None,
        sample_key=sample_key,
        backend="torch",
    )
    model = MRVI(
        mrvi_adata,
        n_latent=config["n_latent_z"],
        n_latent_u=config["n_latent_u"],
        encoder_n_hidden=config["encoder_n_hidden"],
        encoder_n_layers=config["encoder_n_layers"],
    )
    model.train(
        max_epochs=config["max_epochs"],
        accelerator="auto",
        devices="auto",
        train_size=config["train_size"],
        validation_size=1.0 - config["train_size"],
        batch_size=config["batch_size"],
        early_stopping=config["early_stopping"],
        early_stopping_patience=config["early_stopping_patience"],
        check_val_every_n_epoch=1,
    )

    latent = np.asarray(
        model.get_latent_representation(
            adata=mrvi_adata,
            batch_size=config["batch_size"],
            use_mean=True,
            give_z=False,
        ),
        dtype=np.float32,
    )
    barcodes = mrvi_adata.obs_names.copy()

    del model, mrvi_adata
    gc.collect()
    torch.cuda.empty_cache()

    return MethodOutput(
        latent=latent,
        barcodes=barcodes,
        representation_key="X_mrvi_u",
    )

## 5. High-Level scRareBench evaluation

In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="MrVI-u",
    runner=run_mrvi,
    config=METHOD_CONFIG,
    dependencies=METHOD_DEPENDENCIES,
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,
        "write_pdf_report": False,
        "include_latent_in_bundle": False,
        "include_cell_ids": False,
    },
    install_dependencies=False,
    finalize=True,
    copy_input_per_seed=False,
    overwrite=True,
)

## 6. Results

In [ ]:
from IPython.display import display

display(result.summary().round(5))
print("Seeds:", result.seeds)
print("Interactive report:", result.report_path)
print("Final ZIP:", result.archive_path)
print("Summary JSON:", result.summary_path)

## 7. Download ZIP to your computer

In [ ]:
from pathlib import Path
from google.colab import files

archive = Path(result.archive_path)
print(f"Downloading {archive.name} ({archive.stat().st_size / 1024**2:.1f} MB)")
files.download(str(archive))